In [1]:

import cv2
import numpy as np


# pixel-to-centimeter calibration

In [2]:

cap = cv2.VideoCapture(1)

cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1200)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
ret, prev = cap.read()
prev=cv2.rotate(prev,cv2.ROTATE_90_CLOCKWISE)


if not ret:
    print("Camera not available")
    cap.release()
    exit()

prev_gray = cv2.cvtColor(prev, cv2.COLOR_BGR2GRAY)
prev_gray = cv2.GaussianBlur(prev_gray, (5, 5), 0)

kernel = np.ones((5, 5), np.uint8)

reference = (20, 200)

# Calibration: measure a known length in your camera image
known_length_cm = 10.0
known_length_px = 200.0

cm_per_pixel = known_length_cm / known_length_px

try:
    while cap.isOpened():

        ret, frame = cap.read()

        if not ret:
            break

        frame=cv2.rotate(frame,cv2.ROTATE_90_CLOCKWISE)

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        gray = cv2.GaussianBlur(gray, (5, 5), 0)

        diff = cv2.absdiff(prev_gray, gray)

        _, thresh = cv2.threshold(diff, 25, 255, cv2.THRESH_BINARY)

        opened = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel)
        cleaned = cv2.morphologyEx(opened, cv2.MORPH_CLOSE, kernel, iterations=2)

        contours, _ = cv2.findContours(cleaned, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        cv2.circle(frame, reference, 5, (0, 0, 255), -1)

        for contour in contours:

            if cv2.contourArea(contour) < 10000:
                continue

            x, y, w, h = cv2.boundingRect(contour)

            (cx, cy), radius = cv2.minEnclosingCircle(contour)
            center = (int(cx), int(cy))

            distance_px = np.linalg.norm(np.array(center) - np.array(reference))
            distance_cm = distance_px * cm_per_pixel

            cv2.circle(frame, center, int(radius), (0, 255, 0), 2)
            cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)

            text = f"Distance: {distance_cm:.1f} cm"

            (text_w, text_h), baseline = cv2.getTextSize(
                text, cv2.FONT_HERSHEY_SIMPLEX, 0.8, 2)

            text_x = x
            text_y = max(y - 12, text_h + 10)

            cv2.rectangle(
                frame,
                (text_x, text_y - text_h - 8),
                (text_x + text_w + 8, text_y + baseline),(0, 0, 0), -1 )

            cv2.putText(
                frame, text, (text_x + 4, text_y),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2
            )

            cv2.line(frame, reference, center, (255, 0, 0), 2)

        cv2.imshow("Motion Detection", frame)
        cv2.imshow("Motion Mask", cleaned)

        prev_gray = gray.copy()

        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

finally:
    cap.release()
    cv2.destroyAllWindows()